# FlyFFN v2 vs SmolLM2-135M — progressive FFN-only replacement

Standard SmolLM2 attention stays unchanged. Non-anchor FFNs start **exactly dense**, then attempt the progressive schedule **8→6→4→3→2 shards**. Every stage is accepted only when the full-model CE gap stays inside the configured quality gate; otherwise that group is rolled back. Every fourth FFN remains a dense anchor.

This is a quality prototype: dense/sparse blending still computes all shards. A fused selected-shard kernel is a later speed optimization.

In [1]:
#@title 1. Update repository, install, and syntax-check
import pathlib, subprocess, sys
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),
                'transformers>=4.56','datasets>=3.0','accelerate>=1.0','huggingface_hub>=0.34',
                'pandas>=2.0','requests>=2.31','tqdm>=4.66'], check=True)
for p in [REPO_DIR/'src'/'tinycenn_lm'/'smollm2_flyffn_v2.py', REPO_DIR/'scripts'/'run_smollm2_flyffn_v2.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(p)], check=True)
print('✓ FlyFFN-v2 module and runner syntax OK')
print('Ready:', REPO_DIR)


✓ FlyFFN-v2 module and runner syntax OK
Ready: /content/TinyCeNN-LM


In [2]:
#@title 2. Configuration
RUN_MODE = 'quick' #@param ['quick','strong']
SEQ_LEN = 128 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FLY_NODES = 256 #@param {type:'integer'}
ROUTER_RANK = 64 #@param {type:'integer'}
MAX_EDGES = 2048 #@param {type:'integer'}
NUM_SHARDS = 8 #@param {type:'integer'}
GRAPH_STEPS = 1 #@param {type:'integer'}
GRAPH_MIX_INIT = 0.50 #@param {type:'number'}
ANCHOR_EVERY = 4 #@param {type:'integer'}
MAX_CE_GAP = 0.45 #@param {type:'number'}
RUN_REWIRED_CONTROL = True #@param {type:'boolean'}
OUTPUT_DIR = REPO_DIR/'results'/'flyffn_v2_smollm2_135m'
print('Attention: standard SmolLM2 (unchanged)')
print('Schedule: 8→6→4→3→2 | dense anchors every', ANCHOR_EVERY, 'layers | CE gate <=', MAX_CE_GAP)


Attention: standard SmolLM2 (unchanged)
Schedule: 8→6→4→3→2 | dense anchors every 4 layers | CE gate <= 0.45


In [3]:
#@title 3. Train / evaluate — live output
import os, subprocess, sys
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_smollm2_flyffn_v2.py'),
     '--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),'--batch-size',str(BATCH_SIZE),
     '--fly-nodes',str(FLY_NODES),'--router-rank',str(ROUTER_RANK),'--max-edges',str(MAX_EDGES),
     '--num-shards',str(NUM_SHARDS),'--graph-steps',str(GRAPH_STEPS),'--graph-mix-init',str(GRAPH_MIX_INIT),
     '--anchor-every',str(ANCHOR_EVERY),'--max-ce-gap',str(MAX_CE_GAP),'--output-dir',str(OUTPUT_DIR)]
if RUN_REWIRED_CONTROL: cmd.append('--rewired')
print('='*92); print('FlyFFN-v2 progressive FFN experiment'); print('Command:', ' '.join(cmd)); print('='*92)
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TQDM_MININTERVAL']='1'
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in iter(p.stdout.readline,''):
    print(line, end='', flush=True)
rc=p.wait(); print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


FlyFFN-v2 progressive FFN experiment
Command: /usr/bin/python3 -u /content/TinyCeNN-LM/scripts/run_smollm2_flyffn_v2.py --run-mode quick --seq-len 128 --batch-size 1 --fly-nodes 256 --router-rank 64 --max-edges 2048 --num-shards 8 --graph-steps 1 --graph-mix-init 0.5 --anchor-every 4 --max-ce-gap 0.45 --output-dir /content/TinyCeNN-LM/results/flyffn_v2_smollm2_135m --rewired
DEVICE cuda | dtype=torch.bfloat16 | gpu=NVIDIA L4
STAGE graph: preparing FlyWire topology

FlyWire graph: 0.00B [00:00, ?B/s]
FlyWire graph: 351kB [00:01, 342kB/s]
FlyWire graph: 1.14MB [00:02, 549kB/s]
FlyWire graph: 1.75MB [00:03, 542kB/s]
FlyWire graph: 2.41MB [00:04, 551kB/s]
FlyWire graph: 3.06MB [00:05, 556kB/s]
FlyWire graph: 3.74MB [00:06, 568kB/s]
FlyWire graph: 4.36MB [00:07, 583kB/s]
FlyWire graph: 4.95MB [00:08, 559kB/s]
FlyWire graph: 5.57MB [00:10, 552kB/s]
FlyWire graph: 6.18MB [00:11, 547kB/s]
FlyWire graph: 6.87MB [00:12, 561kB/s]
FlyWire graph: 7.49MB [00:13, 553kB/s]
FlyWire graph: 8.25MB [00:14

In [4]:
#@title 4. Results
import json, pandas as pd
from IPython.display import display
summary=pd.read_csv(OUTPUT_DIR/'summary.csv',index_col=0)
report=json.loads((OUTPUT_DIR/'report.json').read_text())
samples=json.loads((OUTPUT_DIR/'samples.json').read_text())
display(summary)
print('\nCHECKS')
print('Architecture:',report['architecture'])
print('Attention unchanged:',report['attention_unchanged'])
print('FlyFFN-v2 layers:',report['flyffn_layers'])
print('Dense FFN anchors:',report['dense_anchor_layers'])
print('Dense-equivalence max logit diff:',report['dense_equivalence_biological_max_abs_logit_diff'])
print('Device:',report['device'],'| dtype:',report['dtype'])
print('\nKEY METRICS')
for k in ['fly_ce_gap_vs_smollm2','fly_ppl_ratio_vs_smollm2','parameter_ratio_fly_over_smollm2','decode_speed_ratio_fly_over_smollm2','biological_topology_ce_gain','biological_topology_ppl_gain_pct']:
    if k in report: print(k,':',report[k])
print('\nFINAL BIOLOGICAL ROUTING SCHEDULE')
for layer,state in report['biological_routing_schedule'].items(): print(f'layer {layer:>2}: k={state["active_k"]}, mix={state["route_mix"]:.2f}')
print('\nSAMPLES')
for x in samples:
    print('='*90); print('PROMPT:',x['prompt']); print(x['text'])


,ce,perplexity,weight_mb,buffer_mb,prefill_tokens_s,prefill_peak_extra_mb,decode_tokens_s,decode_peak_extra_mb,teacher_kl
SmolLM2-135M,2.722747,15.222075,256.567017,0.000244,3337.394612,14.953125,28.690581,0.89209,NaN
FlyFFN-v2 biological,2.960926,19.315856,261.418755,0.250463,1635.582799,15.042969,15.672170,0.89209,0.403756
FlyFFN-v2 rewired,2.985382,19.794052,NaN,NaN,NaN,NaN,NaN,NaN,0.427231



CHECKS
Architecture: FlyFFN v2: standard SmolLM2 attention + progressive FlyWire sparse FFN + dense anchors
Attention unchanged: True
FlyFFN-v2 layers: 23
Dense FFN anchors: [3, 7, 11, 15, 19, 23, 27]
Dense-equivalence max logit diff: 4.90625
Device: cuda | dtype: torch.bfloat16

KEY METRICS
fly_ce_gap_vs_smollm2 : 0.23817964394887303
fly_ppl_ratio_vs_smollm2 : 1.2689371292273663
parameter_ratio_fly_over_smollm2 : 1.009455108533317
decode_speed_ratio_fly_over_smollm2 : 0.5462479179442093
biological_topology_ce_gain : 0.02445518970489502
biological_topology_ppl_gain_pct : 2.4158584319492156

FINAL BIOLOGICAL ROUTING SCHEDULE
layer  0: k=4, mix=0.50
layer  1: k=4, mix=0.50
layer  2: k=4, mix=0.50
layer  4: k=4, mix=0.50
layer  5: k=4, mix=0.50
layer  6: k=4, mix=0.50
layer  8: k=4, mix=0.25
layer  9: k=4, mix=0.25
layer 10: k=4, mix=0.25
layer 12: k=6, mix=0.10
layer 13: k=6, mix=0.10
layer 14: k=6, mix=0.10
layer 16: k=6, mix=0.10
layer 17: k=6, mix=0.10
layer 18: k=6, mix=0.10
layer 2

In [7]:
# ============================================================
# FAST EVAL — SmolLM2-135M vs FlyFFN-v2
# Fix imports automatically
# ============================================================

import gc
import json
import os
import random
import sys
import subprocess
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display

# ------------------------------------------------------------
# FIX TinyCeNN-LM IMPORT
# ------------------------------------------------------------

REPO_DIR = Path("/content/TinyCeNN-LM")

if not REPO_DIR.exists():
    print("TinyCeNN-LM not found — cloning repository...")
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/vtavakkoli/TinyCeNN-LM.git",
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    print("TinyCeNN-LM found:", REPO_DIR)

# Make sure we have the latest FlyFFN-v2 code
subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "origin"],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "reset",
        "--hard",
        "origin/main",
    ],
    check=True,
)

# Add src/ directly to current Jupyter Python process
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Also install editable version for subprocesses / later cells
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

# Verify exact file exists
MODULE_FILE = (
    SRC_DIR
    / "tinycenn_lm"
    / "smollm2_flyffn_v2.py"
)

assert MODULE_FILE.exists(), (
    f"FlyFFN-v2 module not found: {MODULE_FILE}"
)

print("✓ Repository:", REPO_DIR)
print("✓ Source path:", SRC_DIR)
print("✓ FlyFFN-v2 module:", MODULE_FILE)

# ------------------------------------------------------------
# NOW IMPORT FlyFFN-v2
# ------------------------------------------------------------

from tinycenn_lm.smollm2_flyffn_v2 import (
    FlyFFNV2Config,
    replace_ffns_with_fly_v2,
    assert_flyffn_v2_replacement,
)

print("✓ tinycenn_lm.smollm2_flyffn_v2 imported successfully")

# ============================================================
# CONTINUE YOUR EXISTING FAST EVAL CODE BELOW
# ============================================================

TinyCeNN-LM found: /content/TinyCeNN-LM
✓ Repository: /content/TinyCeNN-LM
✓ Source path: /content/TinyCeNN-LM/src
✓ FlyFFN-v2 module: /content/TinyCeNN-LM/src/tinycenn_lm/smollm2_flyffn_v2.py
✓ tinycenn_lm.smollm2_flyffn_v2 imported successfully


In [8]:
# ============================================================
# FAST EVAL — SmolLM2-135M vs trained FlyFFN-v2
# 50 samples each:
#   MMLU-Pro / PIQA / MMMLU-DE / GPQA-Diamond
# ============================================================

import gc
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display

from tinycenn_lm.smollm2_flyffn_v2 import (
    FlyFFNV2Config,
    replace_ffns_with_fly_v2,
    assert_flyffn_v2_replacement,
)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

BASE_MODEL = "HuggingFaceTB/SmolLM2-135M"
N = 50
BATCH_SIZE_EVAL = 8
MAX_LENGTH = 768
SEED = 2026

# Reuse OUTPUT_DIR from the FlyFFN-v2 notebook if available
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path(
        "/content/TinyCeNN-LM/results/flyffn_v2_smollm2_135m"
    )

OUTPUT_DIR = Path(OUTPUT_DIR)
REPORT_FILE = OUTPUT_DIR / "report.json"
CHECKPOINT_FILE = OUTPUT_DIR / "biological_flyffn_v2.pt"

assert REPORT_FILE.exists(), f"Missing: {REPORT_FILE}"
assert CHECKPOINT_FILE.exists(), f"Missing: {CHECKPOINT_FILE}"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if device.type == "cuda":
    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
else:
    dtype = torch.float32

print("=" * 90)
print("FAST EVAL: SmolLM2-135M vs FlyFFN-v2")
print("=" * 90)
print("Device :", device)
print("dtype  :", dtype)

if device.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

print(
    "Items  :",
    {
        "MMLU-Pro": N,
        "PIQA": N,
        "MMMLU-DE": N,
        "GPQA-Diamond": N,
    },
)

# ------------------------------------------------------------
# HF TOKEN — only required if GPQA is gated
# ------------------------------------------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    if not HF_TOKEN:
        try:
            HF_TOKEN = userdata.get("HF_TOKEN")
        except Exception:
            pass
except Exception:
    pass

# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

LETTERS = list("ABCDEFGHIJ")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def sample_dataset(ds, n, seed):
    n = min(n, len(ds))
    return ds.shuffle(seed=seed).select(range(n))


def make_prompt(question, options, german=False):
    labels = LETTERS[:len(options)]

    if german:
        intro = (
            "Wähle die richtige Antwort. "
            "Antworte nur mit dem Buchstaben.\n\n"
        )
        qword = "Frage"
        aword = "Antwort"
    else:
        intro = (
            "Choose the correct answer. "
            "Reply only with the answer letter.\n\n"
        )
        qword = "Question"
        aword = "Answer"

    lines = [
        intro + f"{qword}: {question}",
        "",
    ]

    for label, option in zip(labels, options):
        lines.append(f"{label}. {option}")

    lines += ["", f"{aword}:"]

    return "\n".join(lines), labels


# Cache candidate token IDs
_label_id_cache = {}

def get_label_ids(labels):
    key = tuple(labels)

    if key in _label_id_cache:
        return _label_id_cache[key]

    ids = []

    for label in labels:
        # Prefer the token after a space because the prompt ends in ":"
        candidates = [
            tokenizer.encode(
                " " + label,
                add_special_tokens=False,
            ),
            tokenizer.encode(
                label,
                add_special_tokens=False,
            ),
        ]

        token_id = None

        for candidate in candidates:
            if len(candidate) == 1:
                token_id = candidate[0]
                break

        if token_id is None:
            raise RuntimeError(
                f"Label {label!r} is not a single token "
                f"for this tokenizer: {candidates}"
            )

        ids.append(token_id)

    _label_id_cache[key] = ids
    return ids


# ------------------------------------------------------------
# LOAD DATASETS
# ------------------------------------------------------------

benchmarks = {}

print("\n[DATA] MMLU-Pro ...")

ds = load_dataset(
    "TIGER-Lab/MMLU-Pro",
    split="test",
)

ds = sample_dataset(ds, N, SEED)

items = []

for ex in ds:
    options = list(ex["options"])

    prompt, labels = make_prompt(
        ex["question"],
        options,
    )

    items.append({
        "prompt": prompt,
        "labels": labels,
        "gold": int(ex["answer_index"]),
    })

benchmarks["MMLU-Pro"] = items

print("       loaded:", len(items))


print("[DATA] PIQA ...")

ds = load_dataset(
    "regisss/piqa",
    split="validation",
)

ds = sample_dataset(ds, N, SEED + 1)

items = []

for ex in ds:
    options = [
        ex["sol1"],
        ex["sol2"],
    ]

    prompt, labels = make_prompt(
        ex["goal"],
        options,
    )

    items.append({
        "prompt": prompt,
        "labels": labels,
        "gold": int(ex["label"]),
    })

benchmarks["PIQA"] = items

print("       loaded:", len(items))


print("[DATA] MMMLU-DE ...")

# First try explicit DE_DE config.
try:
    ds = load_dataset(
        "openai/MMMLU",
        "DE_DE",
        split="test",
    )

except Exception:
    # Current dataset may expose languages in the default table.
    ds = load_dataset(
        "openai/MMMLU",
        split="test",
    )

    # If language column exists, filter to German.
    cols = ds.column_names

    lang_col = next(
        (
            x
            for x in [
                "Language",
                "language",
                "lang",
                "locale",
            ]
            if x in cols
        ),
        None,
    )

    if lang_col is not None:
        ds = ds.filter(
            lambda x:
                str(x[lang_col]).upper()
                in {"DE", "DE_DE", "GERMAN"}
        )

ds = sample_dataset(ds, N, SEED + 2)

items = []

for ex in ds:
    options = [
        str(ex["A"]),
        str(ex["B"]),
        str(ex["C"]),
        str(ex["D"]),
    ]

    prompt, labels = make_prompt(
        str(ex["Question"]),
        options,
        german=True,
    )

    ans = str(ex["Answer"]).strip().upper()

    items.append({
        "prompt": prompt,
        "labels": labels,
        "gold": labels.index(ans),
    })

benchmarks["MMMLU-DE"] = items

print("       loaded:", len(items))


print("[DATA] GPQA-Diamond ...")

gpqa_available = True

try:
    ds = load_dataset(
        "Idavidrein/gpqa",
        "gpqa_diamond",
        split="train",
        token=HF_TOKEN,
    )

    ds = sample_dataset(ds, N, SEED + 3)

    items = []

    for i, ex in enumerate(ds):

        original = [
            ex["Correct Answer"],
            ex["Incorrect Answer 1"],
            ex["Incorrect Answer 2"],
            ex["Incorrect Answer 3"],
        ]

        # Correct answer must not always be A.
        rng = random.Random(SEED + 10000 + i)

        order = list(range(4))
        rng.shuffle(order)

        options = [
            original[j]
            for j in order
        ]

        gold = order.index(0)

        prompt, labels = make_prompt(
            ex["Question"],
            options,
        )

        items.append({
            "prompt": prompt,
            "labels": labels,
            "gold": gold,
        })

    benchmarks["GPQA-Diamond"] = items

    print("       loaded:", len(items))

except Exception as e:

    gpqa_available = False

    print("       SKIPPED")
    print(
        "       GPQA access is gated. "
        "Accept the dataset terms and add HF_TOKEN "
        "to Colab Secrets."
    )
    print("       Error:", str(e)[:250])


print("\nLoaded FastEval items:")

for name, items in benchmarks.items():
    print(f"  {name:<16}: {len(items)}")


# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

@torch.inference_mode()
def fast_eval_model(model, name):

    model.eval()

    results = {}

    print("\n" + "=" * 90)
    print("MODEL:", name)
    print("=" * 90)

    for bench_name, items in benchmarks.items():

        correct = 0
        total = 0

        start_time = time.perf_counter()

        print(
            f"\n[{name}] {bench_name}"
        )

        for start in range(
            0,
            len(items),
            BATCH_SIZE_EVAL,
        ):

            batch = items[
                start:start + BATCH_SIZE_EVAL
            ]

            prompts = [
                x["prompt"]
                for x in batch
            ]

            enc = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            )

            input_ids = enc.input_ids.to(device)
            attention_mask = enc.attention_mask.to(device)

            # With right padding, last real prompt position
            last_pos = (
                attention_mask.sum(dim=1) - 1
            )

            if device.type == "cuda":
                ctx = torch.autocast(
                    "cuda",
                    dtype=dtype,
                )
            else:
                from contextlib import nullcontext
                ctx = nullcontext()

            with ctx:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=False,
                    return_dict=True,
                )

            logits = outputs.logits.float()

            for bi, item in enumerate(batch):

                pos = int(
                    last_pos[bi].item()
                )

                next_logits = logits[
                    bi,
                    pos,
                ]

                ids = get_label_ids(
                    item["labels"]
                )

                scores = next_logits[
                    torch.tensor(
                        ids,
                        device=next_logits.device,
                    )
                ]

                pred = int(
                    scores.argmax().item()
                )

                if pred == item["gold"]:
                    correct += 1

                total += 1

            done = min(
                start + len(batch),
                len(items),
            )

            # Print every 10 examples
            if (
                done % 10 == 0
                or done == len(items)
            ):

                elapsed = (
                    time.perf_counter()
                    - start_time
                )

                accuracy = (
                    100.0
                    * correct
                    / max(total, 1)
                )

                speed = (
                    done
                    / max(elapsed, 1e-9)
                )

                print(
                    f"  {done:>2}/{len(items)}"
                    f" | correct={correct:>2}"
                    f" | acc={accuracy:5.1f}%"
                    f" | {speed:5.2f} q/s"
                )

        accuracy = correct / total

        results[bench_name] = {
            "correct": correct,
            "total": total,
            "accuracy": accuracy,
        }

        print(
            f"  DONE → "
            f"{correct}/{total} "
            f"= {100*accuracy:.1f}%"
        )

    return results


# ------------------------------------------------------------
# LOAD ORIGINAL SMOLLM2
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("Loading ORIGINAL SmolLM2-135M")
print("=" * 90)

baseline = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=(
        dtype
        if device.type == "cuda"
        else torch.float32
    ),
).to(device)

baseline_results = fast_eval_model(
    baseline,
    "SmolLM2",
)

del baseline

gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# REBUILD TRAINED FlyFFN-v2 BIOLOGICAL
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("Loading trained FlyFFN-v2 BIOLOGICAL")
print("=" * 90)

report = json.loads(
    REPORT_FILE.read_text()
)

state = torch.load(
    CHECKPOINT_FILE,
    map_location="cpu",
    weights_only=True,
)

# Shared biological FlyWire adjacency is inside checkpoint.
adj_keys = [
    k
    for k in state.keys()
    if k.endswith(
        "flyffn_shared_graph.adjacency"
    )
]

if not adj_keys:

    # fallback for alternate state naming
    adj_keys = [
        k
        for k in state.keys()
        if "shared_graph" in k
        and k.endswith("adjacency")
    ]

if not adj_keys:
    raise RuntimeError(
        "Could not find shared FlyWire adjacency "
        "inside biological_flyffn_v2.pt"
    )

bio_adj = state[
    adj_keys[0]
].float()

cfg_data = report["config"]

fly_cfg = FlyFFNV2Config(
    fly_nodes=int(
        cfg_data["fly_nodes"]
    ),
    router_rank=int(
        cfg_data["router_rank"]
    ),
    num_shards=int(
        cfg_data["num_shards"]
    ),
    graph_steps=int(
        cfg_data["graph_steps"]
    ),
    graph_mix_init=float(
        cfg_data["graph_mix_init"]
    ),
    anchor_every=int(
        cfg_data["anchor_every"]
    ),
)

fly_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=(
        dtype
        if device.type == "cuda"
        else torch.float32
    ),
).to(device)

replace_ffns_with_fly_v2(
    fly_model,
    fly_cfg,
    bio_adj,
)

incompatible = fly_model.load_state_dict(
    state,
    strict=False,
)

important_missing = [
    k
    for k in incompatible.missing_keys
    if ".mlp." in k
]

if important_missing:
    raise RuntimeError(
        "Missing FlyFFN-v2 parameters: "
        + str(important_missing[:10])
    )

assert_flyffn_v2_replacement(
    fly_model,
    fly_cfg.anchor_every,
)

print("FlyFFN-v2 checkpoint loaded ✓")


# Show actual final routing state
print("\nFinal routing state from training:")

schedule = report.get(
    "biological_routing_schedule",
    {},
)

for layer, x in schedule.items():
    print(
        f"  layer {int(layer):>2}: "
        f"k={x['active_k']} "
        f"mix={x['route_mix']:.2f}"
    )


fly_results = fast_eval_model(
    fly_model,
    "FlyFFN-v2",
)


# ------------------------------------------------------------
# FINAL JUPYTER TABLE
# ------------------------------------------------------------

rows = []

for bench_name in benchmarks.keys():

    b = baseline_results[
        bench_name
    ]

    f = fly_results[
        bench_name
    ]

    base_pct = 100 * b["accuracy"]
    fly_pct = 100 * f["accuracy"]

    rows.append({
        "Benchmark": bench_name,

        "SmolLM2 correct":
            f"{b['correct']}/{b['total']}",

        "SmolLM2 %":
            base_pct,

        "FlyFFN-v2 correct":
            f"{f['correct']}/{f['total']}",

        "FlyFFN-v2 %":
            fly_pct,

        "Δ FlyFFN":
            fly_pct - base_pct,
    })


result_df = pd.DataFrame(
    rows
).set_index("Benchmark")


# Macro average
macro_base = result_df[
    "SmolLM2 %"
].mean()

macro_fly = result_df[
    "FlyFFN-v2 %"
].mean()

macro_delta = (
    macro_fly - macro_base
)


print("\n")
print("=" * 90)
print("FAST EVAL FINAL RESULT")
print("=" * 90)

display(
    result_df.style.format({
        "SmolLM2 %": "{:.1f}%",
        "FlyFFN-v2 %": "{:.1f}%",
        "Δ FlyFFN": "{:+.1f}",
    })
)

print(
    f"\nMacro average:"
    f"\n  SmolLM2   = {macro_base:.1f}%"
    f"\n  FlyFFN-v2 = {macro_fly:.1f}%"
    f"\n  Difference = {macro_delta:+.1f} points"
)

print(
    "\nNote: FastEval uses 50 deterministic "
    "zero-shot multiple-choice examples per benchmark. "
    "It is intended for quick model comparison, "
    "not an official leaderboard score."
)

FAST EVAL: SmolLM2-135M vs FlyFFN-v2
Device : cuda
dtype  : torch.bfloat16
GPU    : NVIDIA L4
Items  : {'MMLU-Pro': 50, 'PIQA': 50, 'MMMLU-DE': 50, 'GPQA-Diamond': 50}

[DATA] MMLU-Pro ...


README.md:   0%|          | 0.00/11.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 42.9kB            

...


MODEL: SmolLM2

[SmolLM2] MMLU-Pro
  40/50 | correct= 4 | acc= 10.0% | 58.58 q/s
  50/50 | correct= 4 | acc=  8.0% | 64.80 q/s
  DONE → 4/50 = 8.0%

[SmolLM2] PIQA
  40/50 | correct=17 | acc= 42.5% | 196.23 q/s
  50/50 | correct=21 | acc= 42.0% | 173.00 q/s
  DONE → 21/50 = 42.0%

[SmolLM2] MMMLU-DE
  40/50 | correct=10 | acc= 25.0% | 104.65 q/s
  50/50 | correct=12 | acc= 24.0% | 107.39 q/s
  DONE → 12/50 = 24.0%

Loading trained FlyFFN-v2 BIOLOGICAL


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

FlyFFN-v2 checkpoint loaded ✓

Final routing state from training:
  layer  0: k=4 mix=0.50
  layer  1: k=4 mix=0.50
  layer  2: k=4 mix=0.50
  layer  4: k=4 mix=0.50
  layer  5: k=4 mix=0.50
  layer  6: k=4 mix=0.50
  layer  8: k=4 mix=0.25
  layer  9: k=4 mix=0.25
  layer 10: k=4 mix=0.25
  layer 12: k=6 mix=0.10
  layer 13: k=6 mix=0.10
  layer 14: k=6 mix=0.10
  layer 16: k=6 mix=0.10
  layer 17: k=6 mix=0.10
  layer 18: k=6 mix=0.10
  layer 20: k=6 mix=0.10
  layer 21: k=6 mix=0.10
  layer 22: k=6 mix=0.10
  layer 24: k=4 mix=0.25
  layer 25: k=4 mix=0.25
  layer 26: k=4 mix=0.25
  layer 28: k=6 mix=0.10
  layer 29: k=6 mix=0.10

MODEL: FlyFFN-v2

[FlyFFN-v2] MMLU-Pro
  40/50 | correct= 4 | acc= 10.0% | 70.90 q/s
  50/50 | correct= 5 | acc= 10.0% | 68.87 q/s
  DONE → 5/50 = 10.0%

[FlyFFN-v2] PIQA
  40/50 | correct=21 | acc= 52.5% | 110.15 q/s
  50/50 | correct=25 | acc= 50.0% | 98.69 q/s
  DONE → 25/50 = 50.0%

[FlyFFN-v2] MMMLU-DE
  40/50 | correct=13 | acc= 32.5% | 64.08 q/s
  5

,SmolLM2 correct,SmolLM2 %,FlyFFN-v2 correct,FlyFFN-v2 %,Δ FlyFFN
Benchmark,,,,,
MMLU-Pro,4/50,8.0%,5/50,10.0%,+2.0
PIQA,21/50,42.0%,25/50,50.0%,+8.0
MMMLU-DE,12/50,24.0%,16/50,32.0%,+8.0



Macro average:
  SmolLM2   = 24.7%
  FlyFFN-v2 = 30.7%
  Difference = +6.0 points

Note: FastEval uses 50 deterministic zero-shot multiple-choice examples per benchmark. It is intended for quick model comparison, not an official leaderboard score.


In [10]:
# ============================================================
# SAVE FlyFFN-v2 + FASTEVAL TO HUGGING FACE — COMPACT VERSION
# ============================================================

import os, json, shutil, subprocess, sys
from pathlib import Path
import torch

HF_REPO_ID = "vtava/SmolLM2-135M-FlyFFN-v2"

REPO_DIR   = Path("/content/TinyCeNN-LM")
OUTPUT_DIR = REPO_DIR / "results/flyffn_v2_smollm2_135m"
EXPORT_DIR = Path("/content/FlyFFN-v2-HF")

# ------------------------------------------------------------
# 1. Install / login
# ------------------------------------------------------------
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "huggingface_hub", "safetensors"],
    check=True
)

from huggingface_hub import HfApi, notebook_login

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from google.colab import userdata
    if not HF_TOKEN:
        HF_TOKEN = userdata.get("HF_TOKEN")
except:
    pass

if not HF_TOKEN:
    notebook_login()

api = HfApi(token=HF_TOKEN)

# ------------------------------------------------------------
# 2. Check model from FastEval cell
# ------------------------------------------------------------
assert "fly_model" in globals(), \
    "Run the FlyFFN-v2 FastEval/model-loading cell first."

assert "tokenizer" in globals(), \
    "Tokenizer not found."

# ------------------------------------------------------------
# 3. Prepare export folder
# ------------------------------------------------------------
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

EXPORT_DIR.mkdir(parents=True)

print("Saving FlyFFN-v2...")

# Main model + tokenizer
fly_model.save_pretrained(
    EXPORT_DIR,
    safe_serialization=True
)

tokenizer.save_pretrained(
    EXPORT_DIR
)

print("✓ Model + tokenizer")

# ------------------------------------------------------------
# 4. Copy experiment files
# ------------------------------------------------------------
files = [
    "report.json",
    "summary.csv",
    "samples.json",
    "biological_flyffn_v2.pt",
    "bio_progressive_calibration.csv",
    "bio_training_history.csv",
    "rewired_progressive_calibration.csv",
    "rewired_training_history.csv",
]

for name in files:
    src = OUTPUT_DIR / name
    if src.exists():
        shutil.copy2(src, EXPORT_DIR / name)
        print("✓", name)

# Custom architecture source
src = (
    REPO_DIR / "src" /
    "tinycenn_lm" /
    "smollm2_flyffn_v2.py"
)

if src.exists():
    shutil.copy2(
        src,
        EXPORT_DIR / "smollm2_flyffn_v2.py"
    )
    print("✓ custom FlyFFN-v2 code")

# ------------------------------------------------------------
# 5. Save FastEval
# ------------------------------------------------------------
if "result_df" in globals():

    result_df.to_csv(
        EXPORT_DIR / "fast_eval_50.csv"
    )

    fast_eval = {
        "results":
            result_df.reset_index().to_dict(
                orient="records"
            )
    }

    if (
        "macro_base" in globals()
        and "macro_fly" in globals()
    ):
        fast_eval["macro"] = {
            "SmolLM2": float(macro_base),
            "FlyFFN-v2": float(macro_fly),
            "delta": float(
                macro_fly - macro_base
            )
        }

    (
        EXPORT_DIR /
        "fast_eval_50.json"
    ).write_text(
        json.dumps(
            fast_eval,
            indent=2
        )
    )

    print("✓ FastEval results")

# ------------------------------------------------------------
# 6. Small model card — no huge triple string
# ------------------------------------------------------------
report = {}

report_file = OUTPUT_DIR / "report.json"

if report_file.exists():
    report = json.loads(
        report_file.read_text()
    )

bio = report.get("biological", {})

ce_base = bio.get("teacher_ce", "N/A")
ce_fly  = bio.get("ce", "N/A")
ppl_base = bio.get(
    "teacher_perplexity",
    "N/A"
)
ppl_fly = bio.get(
    "perplexity",
    "N/A"
)

anchors = report.get(
    "dense_anchor_layers",
    []
)

schedule = report.get(
    "biological_routing_schedule",
    {}
)

readme = "\n".join([
    "---",
    "library_name: transformers",
    "pipeline_tag: text-generation",
    "base_model: HuggingFaceTB/SmolLM2-135M",
    "tags:",
    "- smollm2",
    "- flywire",
    "- flyffn",
    "- sparse-ffn",
    "- experimental",
    "---",
    "",
    "# SmolLM2-135M FlyFFN-v2",
    "",
    "Experimental FlyWire-routed progressive sparse FFN version of SmolLM2-135M.",
    "",
    "## Architecture",
    "",
    "- Original SmolLM2 attention is unchanged.",
    "- FFN conversion starts from the exact pretrained dense FFN.",
    "- Progressive routing: 8 → 6 → 4 → 3 → 2 shards.",
    "- CE quality gates with automatic rollback.",
    f"- Dense FFN anchors: `{anchors}`",
    "",
    "## Language-model evaluation",
    "",
    "| Metric | SmolLM2 | FlyFFN-v2 |",
    "|---|---:|---:|",
    f"| CE | {ce_base} | {ce_fly} |",
    f"| Perplexity | {ppl_base} | {ppl_fly} |",
    "",
    "## FastEval",
    "",
    "See `fast_eval_50.csv` and `fast_eval_50.json`.",
    "",
    "Benchmarks:",
    "- MMLU-Pro",
    "- PIQA",
    "- MMMLU-DE",
    "- GPQA-Diamond",
    "",
    "50 deterministic zero-shot samples are used per benchmark.",
    "",
    "## Reproduction",
    "",
    "Custom implementation:",
    "`smollm2_flyffn_v2.py`",
    "",
    "Original experiment checkpoint:",
    "`biological_flyffn_v2.pt`",
    "",
    "Source repository:",
    "https://github.com/vtavakkoli/TinyCeNN-LM",
])

(
    EXPORT_DIR / "README.md"
).write_text(
    readme,
    encoding="utf-8"
)

print("✓ README.md")

# Routing schedule separately
(
    EXPORT_DIR /
    "routing_schedule.json"
).write_text(
    json.dumps(
        schedule,
        indent=2
    )
)

# ------------------------------------------------------------
# 7. Upload
# ------------------------------------------------------------
print("\nCreating Hugging Face repo...")

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True
)

print("Uploading files...")

api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(EXPORT_DIR),
    commit_message=(
        "Upload FlyFFN-v2 model and evaluation results"
    )
)

# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("✓ UPLOAD COMPLETE")
print("=" * 80)

print(
    f"https://huggingface.co/{HF_REPO_ID}"
)

print("\nUploaded:")

for f in sorted(EXPORT_DIR.iterdir()):
    if f.is_file():
        print(
            f"✓ {f.name:<42}"
            f"{f.stat().st_size/1024**2:7.2f} MB"
        )

Saving FlyFFN-v2...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model + tokenizer
✓ report.json
✓ summary.csv
✓ samples.json
✓ biological_flyffn_v2.pt
✓ bio_progressive_calibration.csv
✓ bio_training_history.csv
✓ rewired_progressive_calibration.csv
✓ rewired_training_history.csv
✓ custom FlyFFN-v2 code
✓ FastEval results
✓ README.md

Creating Hugging Face repo...
Uploading files...

✓ UPLOAD COMPLETE
https://huggingface.co/vtava/SmolLM2-135M-FlyFFN-v2

Uploaded:
✓ README.md                                    0.00 MB
✓ bio_progressive_calibration.csv              0.00 MB
✓ bio_training_history.csv                     0.06 MB
✓ biological_flyffn_v2.pt                    157.05 MB
✓ config.json                                  0.00 MB
✓ fast_eval_50.csv                             0.00 MB
✓ fast_eval_50.json                            0.00 MB
✓ generation_config.json                       0.00 MB
✓ model.safetensors                          261.71 MB
✓ report.json                                  0.01 MB
✓ rewired_progressive_calibration.csv       